# MoP + DivPO Co-Author SFT Training

Trains one LoRA adapter per writing-partner persona using the co-author SFT format.
Target behavior: structured pre-writing help — diagnosis, idea, rationale, next step.

Before running, push local data to HuggingFace:
```bash
PYTHONPATH=src python3 -m mop_divpo.cli build-coauthor-data --output-dir data/processed/coauthor_sft
```
Then upload to `DasonTio/mop-divpo-coauthor-sft-data` (already done if using this notebook).

In [ ]:
# Cell 1 - Install dependencies
!pip install -q -U trl peft transformers datasets accelerate huggingface_hub pydantic torchao bitsandbytes

In [ ]:
# Cell 2 - Check GPU
import torch

assert torch.cuda.is_available(), 'No GPU detected. Set Runtime > Change runtime type > T4 GPU.'
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
# Cell 2b - Model size selector
# Set USE_7B = True to train Qwen2.5-7B with 4-bit QLoRA (better quality, ~3x slower)
# Set USE_7B = False for Qwen2.5-0.5B in float16 (fast, proof-of-concept)
USE_7B = False

if USE_7B:
    BASE_MODEL_OVERRIDE = 'Qwen/Qwen2.5-7B-Instruct'
    print('Mode: 7B QLoRA (4-bit) — ~3-4 hrs on T4')
else:
    BASE_MODEL_OVERRIDE = 'Qwen/Qwen2.5-0.5B-Instruct'
    print('Mode: 0.5B float16 — ~1-1.5 hrs on T4')

In [ ]:
# Cell 3 - HuggingFace login (needed to push adapters)
from huggingface_hub import login

login()

In [ ]:
# Cell 4 - Load co-author SFT data from HuggingFace
# Option A (default): load from HF dataset repo
# Option B: upload JSONL files manually into /content/coauthor_sft and skip this cell
import json
from pathlib import Path
from datasets import load_dataset

PERSONA_IDS = ('contrarian', 'cross_domain_analogist', 'systems_thinker', 'minimalist')
HF_DATASET = 'DasonTio/mop-divpo-coauthor-sft-data'
DATA_DIR = Path('/content/coauthor_sft')
DATA_DIR.mkdir(parents=True, exist_ok=True)

try:
    ds = load_dataset(HF_DATASET)
    for persona, split in ds.items():
        out = DATA_DIR / f'{persona}.jsonl'
        with out.open('w') as f:
            for row in split:
                f.write(json.dumps(row, ensure_ascii=False) + '\n')
        print(f'{persona}: {len(split)} rows -> {out}')
except Exception as exc:
    print('Could not load HF dataset. Upload JSONL files manually to', DATA_DIR)
    print(type(exc).__name__, exc)

In [ ]:
# Cell 5 - Training utilities (supports 0.5B float16 and 7B 4-bit QLoRA)
import json
from pathlib import Path
import torch
from datasets import Dataset
from peft import LoraConfig, TaskType
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTConfig, SFTTrainer

BASE_MODEL = BASE_MODEL_OVERRIDE  # set in Cell 2b
HF_ADAPTER_PREFIX = 'DasonTio/mop-divpo-coauthor'


def read_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]


def format_records(records, tokenizer):
    formatted = []
    for row in records:
        messages = row.get('messages')
        if not isinstance(messages, list) or len(messages) < 3:
            continue
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        formatted.append({'text': text})
    return formatted


def load_model_for_training(base_model: str, use_7b: bool):
    if use_7b:
        from transformers import BitsAndBytesConfig
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type='nf4',
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True,
        )
        return AutoModelForCausalLM.from_pretrained(
            base_model,
            quantization_config=bnb_config,
            device_map='auto',
        )
    else:
        return AutoModelForCausalLM.from_pretrained(
            base_model,
            dtype=torch.float16,
            device_map='auto',
        )


def train_persona(persona, data_path, output_dir, max_steps=500, push_to_hub=True):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    records = read_jsonl(data_path)
    formatted = format_records(records, tokenizer)
    assert formatted, f'No usable co-author message rows in {data_path}'
    print(persona, 'training rows:', len(formatted))

    dataset = Dataset.from_list(formatted)
    model = load_model_for_training(BASE_MODEL, USE_7B)

    # 7B: add more target modules; 0.5B: q_proj + v_proj sufficient
    target_modules = ['q_proj', 'v_proj', 'k_proj', 'o_proj'] if USE_7B else ['q_proj', 'v_proj']
    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules=target_modules,
        bias='none',
    )
    args = SFTConfig(
        output_dir=str(output_dir),
        max_steps=max_steps,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=2,
        learning_rate=2e-4,
        warmup_steps=50,
        fp16=not USE_7B,     # float16 for 0.5B on CUDA
        bf16=USE_7B,         # bfloat16 for 7B QLoRA
        logging_steps=10,
        save_steps=max_steps,
        save_total_limit=1,
        report_to='none',
        dataset_text_field='text',
    )
    trainer = SFTTrainer(model=model, args=args, train_dataset=dataset, peft_config=lora_config)
    trainer.train()
    trainer.save_model(str(output_dir))
    tokenizer.save_pretrained(str(output_dir))

    if push_to_hub:
        repo_id = f'{HF_ADAPTER_PREFIX}-{persona}-sft'
        trainer.model.push_to_hub(repo_id)
        tokenizer.push_to_hub(repo_id)
        print('Pushed:', repo_id)

    del model, trainer
    torch.cuda.empty_cache()

In [ ]:
# Cell 6 - Train all personas
OUTPUT_DIR = Path('/content/coauthor_adapters')
MAX_STEPS = 200

for persona in PERSONA_IDS:
    data_path = DATA_DIR / f'{persona}.jsonl'
    if not data_path.exists():
        print('Skipping missing data:', data_path)
        continue
    print(f'\n=== Training {persona} ===')
    train_persona(persona, data_path, OUTPUT_DIR / persona, max_steps=MAX_STEPS, push_to_hub=True)

print('Co-author adapter training complete.')

In [ ]:
# Cell 6b - Download all adapters as a zip (alternative to HF Hub)
# Run this if you want the adapter files directly on your machine.
import shutil
from google.colab import files

zip_path = '/content/coauthor_adapters.zip'
shutil.make_archive('/content/coauthor_adapters', 'zip', '/content/coauthor_adapters')
files.download(zip_path)
print('Download started:', zip_path)

In [ ]:
# Cell 6c - Generate DivPO candidate pairs from trained SFT adapters
# For each persona: generate N=4 responses per prompt, score rarity, select chosen/rejected pair
import json
from pathlib import Path
from collections import Counter

DIVPO_PAIRS_DIR = Path('/content/divpo_pairs')
DIVPO_PAIRS_DIR.mkdir(parents=True, exist_ok=True)
N_CANDIDATES = 4
MAX_PROMPTS = 50   # keep short to fit Colab session time


def lexical_rarity_scores(responses):
    scores = []
    for i, resp in enumerate(responses):
        others = responses[:i] + responses[i+1:]
        tokens = set(resp.lower().split())
        other_tokens = set(' '.join(others).lower().split())
        overlap = len(tokens & other_tokens) / len(tokens) if tokens and others else 0
        uniq = len(set(resp.lower().split())) / max(len(resp.split()), 1)
        scores.append(max(0.0, uniq - overlap))
    mx = max(scores) if scores else 0
    return [s / mx if mx else 0.0 for s in scores]


def generate_candidates(prompt, system_content, tokenizer, model, n, max_new_tokens=256):
    device = next(model.parameters()).device
    messages = [{'role': 'system', 'content': system_content},
                {'role': 'user', 'content': prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(device)
    responses = []
    for _ in range(n):
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                                 do_sample=True, temperature=0.9, top_p=0.9,
                                 pad_token_id=tokenizer.eos_token_id)
        responses.append(tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True))
    return responses


PERSONA_SYSTEM_PROMPTS = {
    'contrarian': 'You are a Contrarian Analyst and writing partner. Identify hidden assumptions and argue minority positions.',
    'cross_domain_analogist': 'You are a Cross-Domain Analogist and writing partner. Find structural isomorphisms between distant domains.',
    'systems_thinker': 'You are a Systems Thinker and writing partner. Map causes, feedback loops, and leverage points.',
    'minimalist': 'You are a Minimalist Designer and writing partner. Identify the single most essential element and design around constraint.',
}

for persona in PERSONA_IDS:
    print(f'\n=== DivPO candidates: {persona} ===')
    adapter_path = OUTPUT_DIR / persona
    if not adapter_path.exists():
        print(f'  No adapter at {adapter_path}, skipping.')
        continue

    data_path = DATA_DIR / f'{persona}.jsonl'
    if not data_path.exists():
        print(f'  No data at {data_path}, skipping.')
        continue

    from peft import PeftModel
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=torch.float16, device_map='auto')
    model = PeftModel.from_pretrained(base, str(adapter_path))
    model.eval()

    rows = [json.loads(l) for l in data_path.read_text().splitlines() if l.strip()]
    prompts = []
    for row in rows:
        for msg in row.get('messages', []):
            if msg.get('role') == 'user':
                prompts.append(msg['content'])
                break
    prompts = prompts[:MAX_PROMPTS]

    pairs = []
    system_content = PERSONA_SYSTEM_PROMPTS[persona]
    for i, prompt in enumerate(prompts):
        print(f'  [{i+1}/{len(prompts)}] generating {N_CANDIDATES} candidates ...', end='\r')
        responses = generate_candidates(prompt, system_content, tokenizer, model, N_CANDIDATES)
        scores = lexical_rarity_scores(responses)
        scored = sorted(zip(scores, responses), reverse=True)
        if len(scored) >= 2 and scored[0][0] > 0:
            pairs.append({
                'id': f'divpo-{persona}-{i}',
                'persona': persona,
                'prompt': prompt,
                'chosen': scored[0][1],
                'rejected': scored[-1][1],
            })

    out_path = DIVPO_PAIRS_DIR / f'{persona}.jsonl'
    with out_path.open('w') as f:
        for pair in pairs:
            f.write(json.dumps(pair, ensure_ascii=False) + '\n')
    print(f'\n  {len(pairs)} pairs written to {out_path}')

    del model, base
    torch.cuda.empty_cache()

print('\nDivPO pair generation complete.')

In [ ]:
# Cell 6d - DPO training on DivPO pairs
# Starts from SFT adapter, trains with DPO loss on chosen/rejected pairs
from trl import DPOConfig, DPOTrainer
from datasets import Dataset as HFDataset

DIVPO_OUTPUT_DIR = Path('/content/divpo_adapters')
DPO_MAX_STEPS = 100
DPO_BETA = 0.1   # KL penalty coefficient — higher = stay closer to SFT reference

for persona in PERSONA_IDS:
    print(f'\n=== DPO training: {persona} ===')
    pairs_path = DIVPO_PAIRS_DIR / f'{persona}.jsonl'
    sft_adapter_path = OUTPUT_DIR / persona
    if not pairs_path.exists():
        print(f'  No pairs at {pairs_path}, skipping.')
        continue
    if not sft_adapter_path.exists():
        print(f'  No SFT adapter at {sft_adapter_path}, skipping.')
        continue

    pairs = [json.loads(l) for l in pairs_path.read_text().splitlines() if l.strip()]
    if not pairs:
        print(f'  Empty pairs file, skipping.')
        continue
    print(f'  {len(pairs)} DivPO pairs loaded.')

    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Trainable model starts from SFT weights
    base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=torch.float16, device_map='auto')
    model = PeftModel.from_pretrained(base, str(sft_adapter_path), is_trainable=True)

    # Frozen reference = SFT weights (DPO computes KL against this)
    ref_base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=torch.float16, device_map='auto')
    ref_model = PeftModel.from_pretrained(ref_base, str(sft_adapter_path), is_trainable=False)
    ref_model.eval()

    dataset = HFDataset.from_list([
        {'prompt': p['prompt'], 'chosen': p['chosen'], 'rejected': p['rejected']}
        for p in pairs
    ])

    out_dir = DIVPO_OUTPUT_DIR / persona
    out_dir.mkdir(parents=True, exist_ok=True)

    dpo_args = DPOConfig(
        output_dir=str(out_dir),
        max_steps=DPO_MAX_STEPS,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=5e-5,
        beta=DPO_BETA,
        fp16=True,
        logging_steps=10,
        save_steps=DPO_MAX_STEPS,
        save_total_limit=1,
        report_to='none',
        remove_unused_columns=False,
    )
    trainer = DPOTrainer(
        model=model,
        ref_model=ref_model,
        args=dpo_args,
        train_dataset=dataset,
        tokenizer=tokenizer,
    )
    trainer.train()
    trainer.save_model(str(out_dir))
    tokenizer.save_pretrained(str(out_dir))

    repo_id = f'{HF_ADAPTER_PREFIX}-{persona}-divpo'
    trainer.model.push_to_hub(repo_id)
    tokenizer.push_to_hub(repo_id)
    print(f'  Pushed DivPO adapter: {repo_id}')

    del model, ref_model, base, ref_base, trainer
    torch.cuda.empty_cache()

print('\nDPO training complete.')

In [ ]:
# Cell 7 - Smoke test one adapter as a writing partner
from peft import PeftModel

PERSONA = 'contrarian'
ADAPTER_REPO = f'{HF_ADAPTER_PREFIX}-{PERSONA}-sft'
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float16, device_map='auto')
model = PeftModel.from_pretrained(base, ADAPTER_REPO)
model.eval()

messages = [
    {'role': 'system', 'content': 'You are a Contrarian Analyst and writing partner. Help the writer challenge hidden assumptions and find sharper thesis angles.'},
    {'role': 'user', 'content': 'Topic: AI in education\nWriter goal: Find a non-obvious thesis angle\nAudience: university instructors'},
]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors='pt').to(model.device)

with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=256, do_sample=True, temperature=0.8, top_p=0.9, pad_token_id=tokenizer.eos_token_id)

print(tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True))